<a href="https://colab.research.google.com/github/marshallpitts/movie-recommender-svd/blob/main/Milestone_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

In [6]:
import os
import subprocess
import sys

USER = "marshallpitts"
REPO = "movie-recommender-svd"

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            f"https://github.com/{USER}/{REPO}.git",
            REPO,
        ],
        check=True,
    )

sys.path.insert(0, os.path.abspath(f"{REPO}/src"))

from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline
from evaluate import to_matrix
from models import baseline_matrix, rank_k

print("Project functions imported successfully.")

Project functions imported successfully.


Pulled from repository

In [14]:
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip",
    )

    with zipfile.ZipFile("ml-100k.zip") as archive:
        archive.extractall()

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"],
)

train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)

print("Total ratings:", len(ratings))
print("Fitting ratings:", len(fit))
print("Validation ratings:", len(val))
print("Test ratings:", len(test))

display(ratings.head())

Total ratings: 100000
Fitting ratings: 81200
Validation ratings: 8832
Test ratings: 9968


,user,item,rating,time
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Copied from milestone 3, downloading files.

In [15]:
titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
header=None, usecols=[1])[1].to_numpy()

print(len(titles), titles[49])

1682 Star Wars (1977)


loading ratings and movie titles.

In [16]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


From Milestone 3 but we use ratings and not train.

In [17]:
def show_factor(f, n=6):

  v = Vt[f - 1]
  idx = np.where(popular)[0]
  order = idx[np.argsort(v[idx])]
  print(f"Factor {f}, one end:")
  for m in order[:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
  print(f"Factor {f}, other end:")
  for m in order[::-1][:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
show_factor(1)

Factor 1, one end:
 -0.087 Clockwork Orange, A (1971)
 -0.079 Trainspotting (1996)
 -0.072 Leaving Las Vegas (1995)
 -0.072 Pulp Fiction (1994)
 -0.069 Brazil (1985)
 -0.067 Fargo (1996)
Factor 1, other end:
 +0.211 Independence Day (ID4) (1996)
 +0.175 Return of the Jedi (1983)
 +0.149 Raiders of the Lost Ark (1981)
 +0.149 Back to the Future (1985)
 +0.136 Braveheart (1995)
 +0.134 Forrest Gump (1994)


We are sorting the popular movies here.

In [18]:
show_factor(3)

Factor 3, one end:
 -0.137 Independence Day (ID4) (1996)
 -0.131 Star Trek: First Contact (1996)
 -0.130 Rock, The (1996)
 -0.129 Alien (1979)
 -0.121 Monty Python and the Holy Grail (1974)
 -0.119 Aliens (1986)
Factor 3, other end:
 +0.163 Sound of Music, The (1965)
 +0.144 Grease (1978)
 +0.099 E.T. the Extra-Terrestrial (1982)
 +0.087 Mary Poppins (1964)
 +0.085 Jaws 2 (1978)
 +0.084 Snow White and the Seven Dwarfs (1937)


inspects 3rd latent factor.

In [19]:
def find(word):

  for m, t in enumerate(titles):
    if word.lower() in t.lower():
      print(m + 1, t)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


We are finding the movie numbers

In [20]:
my_ratings = {50: 5, 172: 5, 183: 5, 195: 5, 89: 4, 1: 4,

  143: 1, 451: 1, 88: 1, 739: 2}

Random ratings of some movies.

In [21]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


Adds myself as a user.

In [22]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
  return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
  print(f" {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
  print(f" {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
 4.28 Close Shave, A (1995)
 4.26 Wrong Trousers, The (1993)
 4.20 Fargo (1996)
 4.15 Casablanca (1942)
 4.14 Raiders of the Lost Ark (1981)
 4.11 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Godfather, The (1972)
 4.07 Schindler's List (1993)
 4.07 Shawshank Redemption, The (1994)
 4.06 Return of the Jedi (1983)
Baseline recommends:
 4.23 Close Shave, A (1995)
 4.15 Wrong Trousers, The (1993)
 4.13 Schindler's List (1993)
 4.13 Shawshank Redemption, The (1994)
 4.09 Casablanca (1942)
 4.08 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Usual Suspects, The (1995)
 4.06 Good Will Hunting (1997)
 4.01 Titanic (1997)
 4.01 12 Angry Men (1957)
in both lists: 6


Compares the top 10 movie rocommendations generated by my personalized model against a baseline model.

In [23]:
personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
 +0.34 Fargo (1996)
 +0.34 Return of the Jedi (1983)
 +0.25 Star Trek: The Wrath of Khan (1982)
 +0.22 Star Trek: First Contact (1996)
 +0.21 2001: A Space Odyssey (1968)
 +0.20 Raiders of the Lost Ark (1981)
Moved down most for me:
 -0.36 English Patient, The (1996)
 -0.21 Liar Liar (1997)
 -0.19 Dirty Dancing (1987)
 -0.19 Mr. Holland's Opus (1995)
 -0.18 Ghost (1990)
 -0.17 Mrs. Doubtfire (1993)


Subtracts the baseline scores from final predicted scores.